# Fig 5 — E6 ABR composability (wide, 2-col)

Left: 8x3 heat map of avg delivered bitrate (viridis), text-overlaid
with bitrate (top) and n_switches (bottom).
Right: strip of mini-boxplots of max_playhead_gap_ms, one per cell,
row-aligned with the heat map.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "figures"))

import matplotlib.pyplot as plt
import numpy as np

from _data import (
    E6_COL_ORDER,
    E6_ROW_ORDER,
    e6_avg_bitrate_matrix,
    e6_heatmap_matrix,
    load_aggregate,
)
from _style import (
    GOP_DURATION_MS,
    TEXT_WIDTH_IN,
    apply_acm_style,
)

apply_acm_style()

In [ ]:
bitrate_matrix = e6_avg_bitrate_matrix()
switches_matrix = e6_heatmap_matrix("n_switches_mean")
agg = load_aggregate("e6")

In [ ]:
fig = plt.figure(figsize=(TEXT_WIDTH_IN, 3.6), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[2.6, 1.0], wspace=0.08)
ax_heat = fig.add_subplot(gs[0, 0])
ax_strip = fig.add_subplot(gs[0, 1])  # independent Y axis

# Heat map.
masked = np.ma.masked_invalid(bitrate_matrix.values)
im = ax_heat.imshow(masked, cmap="viridis", aspect="auto",
                    interpolation="nearest", origin="upper")
ax_heat.set_xticks(range(len(E6_COL_ORDER)))
ax_heat.set_xticklabels(E6_COL_ORDER, rotation=20, ha="right")
ax_heat.set_yticks(range(len(E6_ROW_ORDER)))
ax_heat.set_yticklabels(E6_ROW_ORDER, fontsize=7)
ax_heat.set_xlabel("Bandwidth profile")
ax_heat.set_ylabel("ABR config")

# Cell text overlays.
for i, row in enumerate(E6_ROW_ORDER):
    for j, col in enumerate(E6_COL_ORDER):
        bitrate = bitrate_matrix.iloc[i, j]
        switches = switches_matrix.iloc[i, j] if not np.isnan(switches_matrix.iloc[i, j]) else 0
        if np.isnan(bitrate):
            ax_heat.text(j, i, "—", ha="center", va="center",
                         color="white", fontsize=7)
            continue
        # Choose readable text color based on cell brightness.
        rgba = im.cmap(im.norm(bitrate))
        luminance = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
        text_color = "white" if luminance < 0.5 else "black"
        ax_heat.text(j, i - 0.18, f"{bitrate:.0f}", ha="center", va="center",
                     color=text_color, fontsize=6.5, weight="bold")
        ax_heat.text(j, i + 0.18, f"sw={switches:.1f}", ha="center", va="center",
                     color=text_color, fontsize=6)

cbar = fig.colorbar(im, ax=ax_heat, shrink=0.85, pad=0.02)
cbar.set_label("avg delivered bitrate (kbps)", rotation=270, labelpad=10)

# Strip of mini-boxplots, one per (row, col), row-aligned with heat map.
strip_data = []
strip_positions = []
for i, config in enumerate(E6_ROW_ORDER):
    for j, profile in enumerate(E6_COL_ORDER):
        cell_id = f"{config}_{profile}"
        gaps = agg[agg["cell_id"] == cell_id]["max_playhead_gap_ms"].values
        if len(gaps) == 0:
            continue
        x_pos = i + (j - 1) * 0.22
        strip_data.append(gaps)
        strip_positions.append(x_pos)

ax_strip.boxplot(
    strip_data, positions=strip_positions, vert=False, widths=0.18,
    patch_artist=True,
    boxprops={"facecolor": "#888888", "alpha": 0.7},
    medianprops={"color": "black", "linewidth": 1.0},
    flierprops={"marker": "+", "markersize": 2},
)
ax_strip.axvline(GOP_DURATION_MS, color="black", linestyle="--",
                 linewidth=0.8, alpha=0.7)
ax_strip.text(GOP_DURATION_MS * 1.05, len(E6_ROW_ORDER) - 0.5,
              "1 GOP", fontsize=6.5, va="center")
ax_strip.set_xlabel("max |playhead gap| (ms)")
xmax_data = max((max(d) for d in strip_data if len(d)), default=1300)
ax_strip.set_xlim(0, max(1300, xmax_data + 100))
# Match the heat map row coordinate system (row 0 at top, row 7 at bottom)
# explicitly — we no longer share Y with the heat map, so set ylim ourselves.
ax_strip.set_ylim(len(E6_ROW_ORDER) - 0.5, -0.5)
# Hide tick LABELS only (not the entire Y axis state, which would propagate
# under a shared axis). Keep ticks at row positions for visual alignment.
ax_strip.set_yticks(range(len(E6_ROW_ORDER)))
ax_strip.set_yticklabels([])
ax_strip.tick_params(axis="y", length=0)  # tickmarks invisible

In [ ]:
fig.savefig(Path.cwd().parent / "figures" / "fig5_e6_composability.pdf")
fig.savefig(Path.cwd().parent / "figures" / "fig5_e6_composability.png", dpi=200)